# RNN Quest

In [ ]:
!pip install --quiet --ignore-installed http://nlp.band/static/pypy/lpnlp-2023.10.2-py3-none-any.whl

In [ ]:
import lpnlp

lab = lpnlp.start(
    email="sofiia.tkach.kn.2021@lpnu.ua",  # <---- Fill
    lab="quest_rnn"
)


Ваше завдання: http://nlp.band/static/quest-rnn/b568d5c1d6ac.zip. Удачі! █




Download the package for the task from the link above ^  
Read the README  
Complete the task  
The task is not dependent on the programming language  
The task is not dependent on the framework. Pytorch, NumPy, Keras, or even no external libraries – anything is possible  
Submit your solution below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import numpy as np

def decode_message(embedding_path, W_h_path, W_h_bias_path, U_h_path, W_y_path, vocab_path, hidden_dim=128):
    # Loading model parameters
    with open(W_h_path, 'r') as f:
        W_h = np.array(json.load(f))
    with open(W_h_bias_path, 'r') as f:
        W_h_bias = np.array(json.load(f))
    with open(U_h_path, 'r') as f:
        U_h = np.array(json.load(f))
    with open(W_y_path, 'r') as f:
        W_y = np.array(json.load(f))
    with open(embedding_path, 'r') as f:
        embedding = np.array(json.load(f))

    # Loading the vocabulary
    with open(vocab_path, 'r') as f:
        vocab = json.load(f)

    # Since vocab is a list, create a dictionary from its indices
    if isinstance(vocab, list):
        vocab = {token: index for index, token in enumerate(vocab)}

    # Creating a reverse dictionary to map indices to tokens
    index_to_token = {index: token for token, index in vocab.items()}

    # Initializing the hidden state
    h_t = np.zeros(hidden_dim)

    # '[' indicates the start of the sequence
    token = '['
    message = []

    while token != ']' and len(message) < 100:
        # Get the index of the current token and use the embedding
        token_idx = vocab[token]
        x_t = embedding[token_idx]

        # Calculate the new hidden state h_t
        h_t = np.tanh(W_h @ x_t + W_h_bias + U_h @ h_t)

        # Calculate the output y_t
        y_t = W_y @ h_t

        # Get the index of the next token using argmax
        next_token_idx = np.argmax(y_t)
        token = index_to_token[next_token_idx]

        # Add the token to the message if it is not the end of the sequence
        if token != ']':
            message.append(token)

    # Join the characters into the decoded message
    decoded_message = ''.join(message)
    return decoded_message


decoded_message = decode_message(
    embedding_path='/content/drive/MyDrive/rnn_quest/embedding.weight.json',
    W_h_path='/content/drive/MyDrive/rnn_quest/W_h.weight.json',
    W_h_bias_path='/content/drive/MyDrive/rnn_quest/W_h.bias.json',
    U_h_path='/content/drive/MyDrive/rnn_quest/U_h.weight.json',
    W_y_path='/content/drive/MyDrive/rnn_quest/W_y.weight.json',
    vocab_path='/content/drive/MyDrive/rnn_quest/vocab.json'
)
print(decoded_message)


Назва треку: https://www.youtube.com/watch?v=006ovKeLwYo


In [ ]:
lab.answer("Safe Passage")

Відповідь правильна ✅
You did it! 🚀 Next step: https://tally.so/r/wMXM0p


##Task2
Here is the task with the star: train a language model with the same architecture as in your version. The model should be overfitted on a single message (any message). After that, pack the weights and send them to me at oleksiy.syvokon@gmail.com :) If I can read it, you'll get 2x points. As with any starred task, this is optional.

In [ ]:
import numpy as np
import json

# Hyperparameters
embedding_dim = 64
hidden_dim = 128
vocab_size = 133
learning_rate = 0.01
num_epochs = 500

# Model parameter initialization
W_h = np.random.randn(hidden_dim, embedding_dim) * 0.01
W_h_bias = np.zeros(hidden_dim)
U_h = np.random.randn(hidden_dim, hidden_dim) * 0.01
W_y = np.random.randn(vocab_size, hidden_dim) * 0.01
embedding = np.random.randn(vocab_size, embedding_dim) * 0.01

# Loading the vocabulary
with open('/content/drive/MyDrive/rnn_quest/vocab.json', 'r') as f:
    vocab = json.load(f)

# If vocab is a list, convert it to a dictionary
if isinstance(vocab, list):
    vocab = {token: index for index, token in enumerate(vocab)}

# Create a reverse dictionary to convert indices to symbols
index_to_token = {index: token for token, index in vocab.items()}


# Forward pass
def forward_pass(input_sequence):
    h_t = np.zeros(hidden_dim)
    outputs, h_states = [], []

    for token in input_sequence:
        token_idx = vocab[token]
        x_t = embedding[token_idx]
        h_t = np.tanh(W_h @ x_t + W_h_bias + U_h @ h_t)
        y_t = W_y @ h_t
        outputs.append(y_t)
        h_states.append(h_t)

    return outputs, h_states

# Loss function (cross-entropy)
def cross_entropy_loss(predictions, targets):
    loss = 0
    for prediction, target in zip(predictions, targets):
        exp_scores = np.exp(prediction - np.max(prediction))
        probs = exp_scores / np.sum(exp_scores)
        loss -= np.log(probs[target] + 1e-9)
    return loss / len(targets)

# Backpropagation
def backward_pass(outputs, h_states, message_indices):
    dW_h, dW_h_bias, dU_h, dW_y, d_embedding = (
        np.zeros_like(W_h), np.zeros_like(W_h_bias),
        np.zeros_like(U_h), np.zeros_like(W_y),
        np.zeros_like(embedding)
    )
    dh_next = np.zeros(hidden_dim)

    for t in reversed(range(len(message_indices) - 1)):
        y_t = outputs[t]
        h_t = h_states[t]

        # Gradient of loss with respect to y_t
        exp_scores = np.exp(y_t - np.max(y_t))
        probs = exp_scores / np.sum(exp_scores)
        probs[message_indices[t+1]] -= 1

        # Gradient for W_y and h_t
        dW_y += np.outer(probs, h_t)
        dh = W_y.T @ probs + dh_next
        dh_raw = (1 - h_t ** 2) * dh

        # Gradients for W_h, U_h, W_h_bias, and embedding
        dW_h += np.outer(dh_raw, embedding[message_indices[t]])
        dW_h_bias += dh_raw
        dU_h += np.outer(dh_raw, h_states[t-1] if t > 0 else np.zeros(hidden_dim))
        d_embedding[message_indices[t]] += W_h.T @ dh_raw

        dh_next = U_h.T @ dh_raw

    # Clipping gradients
    for dparam in [dW_h, dW_h_bias, dU_h, dW_y, d_embedding]:
        np.clip(dparam, -1, 1, out=dparam)

    return dW_h, dW_h_bias, dU_h, dW_y, d_embedding

# Training function
def train(message):
    message_indices = [vocab[token] for token in message]

    for epoch in range(num_epochs):
        outputs, h_states = forward_pass(message)
        loss = cross_entropy_loss(outputs, message_indices[1:] + [vocab[']']])

        # Backward pass
        dW_h, dW_h_bias, dU_h, dW_y, d_embedding = backward_pass(outputs, h_states, message_indices)

        # Update weights
        global W_h, W_h_bias, U_h, W_y, embedding
        W_h -= learning_rate * dW_h
        W_h_bias -= learning_rate * dW_h_bias
        U_h -= learning_rate * dU_h
        W_y -= learning_rate * dW_y
        embedding -= learning_rate * d_embedding

        if epoch % 50 == 0:
            print(f'Epoch {epoch}, Loss: {loss:.4f}')



message = "[Sofia]"
train(message)


Epoch 0, Loss: 4.8903
Epoch 50, Loss: 4.8770
Epoch 100, Loss: 4.6780
Epoch 150, Loss: 1.9253
Epoch 200, Loss: 1.6032
Epoch 250, Loss: 1.2255
Epoch 300, Loss: 0.8832
Epoch 350, Loss: 0.5659
Epoch 400, Loss: 0.3285
Epoch 450, Loss: 0.1835
Модель заоверфітилася на повідомленні: [Sofia]


In [ ]:
decoded_message = decode_message(
    embedding_path='/content/drive/MyDrive/rnn_res/embedding.weight.json',
    W_h_path='/content/drive/MyDrive/rnn_res/W_h.weight.json',
    W_h_bias_path='/content/drive/MyDrive/rnn_res/W_h.bias.json',
    U_h_path='/content/drive/MyDrive/rnn_res/U_h.weight.json',
    W_y_path='/content/drive/MyDrive/rnn_res/W_y.weight.json',
    vocab_path='/content/drive/MyDrive/rnn_res/vocab.json'
)
print(decoded_message)

Sofia
